# Test Different Data
Workflow multi-agentico con **Pydantic `with_structured_output()`** + **generazione PDF sintetico**.
Modello: `google/gemma-4-31b-it`

In [1]:
import os, sys, json
os.chdir("/workspaces/hackaton-UNIBG-repo")
sys.path.insert(0, "src")
from dotenv import load_dotenv; load_dotenv()
print("API Key:", "OK" if os.getenv("OPEN_ROUTER_KEY") else "MANCA")

API Key: OK


## Modelli Pydantic

In [2]:
from typing import Any, Dict, List
from pydantic import BaseModel, Field

class FieldDefinition(BaseModel):
    field: str = Field(description="Nome del campo")
    type: str = Field(description="Tipo: string, number, date, list, object")
    sensitive: bool = Field(description="True se contiene dati personali")
    constraints: str = Field(description="Vincoli del campo")

class DocumentExtraction(BaseModel):
    schema_def: List[FieldDefinition] = Field(description="Schema del documento")
    data: Dict[str, Any] = Field(description="Dati estratti dal documento")

class ValidatedRecords(BaseModel):
    records: List[Dict[str, Any]] = Field(description="Record validati e corretti")

print("Modelli Pydantic definiti")

Modelli Pydantic definiti


## 1. OCR del PDF

In [ ]:
from inferencev4 import extract_text_from_pdf

raw_text = extract_text_from_pdf("dataset/LABORATORIO_ANALISI_CLINICHE.pdf")
print(raw_text[:1200])

## 2. Agente Analizzatore

In [7]:
from inferencev4 import build_llm

llm = build_llm(0.2)
structured_llm = llm.with_structured_output(DocumentExtraction)

prompt = (
    "Analizza il seguente testo OCR estratto da un PDF.\n\n"
    "TESTO OCR:\n" + raw_text + "\n\n"
    "Il tuo compito:\n"
    "1. Deduci la struttura del documento (NON fare assunzioni sul tipo).\n"
    "2. Per ogni campo, indica nome, tipo, se e' sensibile, e eventuali vincoli.\n"
    "3. Estrai i dati dal documento nel formato dedotto.\n\n"
    "Marca come sensitive: true TUTTI i campi con dati personali."
)

extraction: DocumentExtraction = structured_llm.invoke(prompt)

schema_list = [fd.model_dump() for fd in extraction.schema_def]
print("=== SCHEMA ===")
print(json.dumps(schema_list, indent=2))
print("\n=== DATI SORGENTE ===")
print(json.dumps(extraction.data, indent=2))

=== SCHEMA ===
[
  {
    "field": "laboratorio_nome",
    "type": "string",
    "sensitive": false,
    "constraints": "obbligatorio"
  },
  {
    "field": "paziente_nome",
    "type": "string",
    "sensitive": true,
    "constraints": "obbligatorio"
  },
  {
    "field": "paziente_codice_fiscale",
    "type": "string",
    "sensitive": true,
    "constraints": "formato codice fiscale italiano"
  },
  {
    "field": "paziente_data_nascita",
    "type": "date",
    "sensitive": true,
    "constraints": "formato GG/MM/AAAA"
  },
  {
    "field": "paziente_indirizzo",
    "type": "string",
    "sensitive": true,
    "constraints": "nessuno"
  },
  {
    "field": "prelievo_data",
    "type": "date",
    "sensitive": true,
    "constraints": "formato GG/MM/AAAA"
  },
  {
    "field": "medico_richiedente",
    "type": "string",
    "sensitive": true,
    "constraints": "nessuno"
  },
  {
    "field": "laboratorio_indirizzo",
    "type": "string",
    "sensitive": false,
    "constraints": "

## 3. Agente Generatore

In [8]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

N = 1
schema_json = json.dumps(schema_list, indent=2)
source_json = json.dumps(extraction.data, indent=2)

llm2 = build_llm(0.7)
generator = create_agent(
    model=llm2,
    tools=[],
    system_prompt=(
        "Sei un anonimizzatore di dati. Produci ESATTAMENTE " + str(N) + " record. "
        "I campi NON sensibili (quantita', prezzi, totali, IVA, date) "
        "DEVONO rimanere IDENTICI all'originale. "
        "Sostituisci SOLO i campi sensitive (nomi, indirizzi, tax ID, IBAN) con valori fittizi. "
        "NON alterare il numero di items. Output SOLO array JSON."
    ),
)

prompt = (
    "SCHEMA:\n" + schema_json +
    "\n\nDATI ORIGINALI (preserva TUTTI i valori non sensitive):\n" + source_json +
    "\n\nAnonimizza solo i campi sensitive."
)

result = generator.invoke({"messages": [HumanMessage(content=prompt)]})
generated_raw = result["messages"][-1].content
print("Generatore output (primi 500 char):", generated_raw[:500])

Generatore output (primi 500 char): ```json
[
  {
    "laboratorio_nome": "LABORATORIO ANALISI CLINICHE \"VITA NOVA\"",
    "paziente_nome": "Luca Verdi",
    "paziente_codice_fiscale": "VRDLCU80A01F205Z",
    "paziente_data_nascita": "15/03/1980",
    "paziente_indirizzo": "Via delle Betulle 12, 20121 Milano",
    "prelievo_data": "29/05/2026",
    "medico_richiedente": "Dott. Marco Neri",
    "laboratorio_indirizzo": "Via delle Scienze 42, 00100 Roma (RM)",
    "laboratorio_piva": "01234567890",
    "analisi_cliniche": [
      {


## 4. Agente Validatore

In [ ]:
llm3 = build_llm(0.2)
structured_val = llm3.with_structured_output(ValidatedRecords)

prompt = (
    "Valida i seguenti record sintetici contro lo schema fornito.\n\n"
    "SCHEMA e constraints:\n" + schema_json +
    "\n\nRECORD DA VALIDARE:\n" + generated_raw +
    "\n\nPer OGNI record: verifica campi, tipi, constraints. "
    "Correggi errori di calcolo. Rimuovi record invalidi."
)

validation: ValidatedRecords = structured_val.invoke(prompt)
validated = validation.records

print(f"Validati: {len(validated)} record")
print(json.dumps(validated[0], indent=2)[:600])

Validati: 1 record
{
  "status": "VALIDO",
  "analisi": {
    "campi_obbligatori": "Presenti (laboratorio_nome, paziente_nome)",
    "tipi_dato": "Corretti",
    "constraints": {
      "paziente_codice_fiscale": "Valido (formato 16 caratteri alfanumerici)",
      "paziente_data_nascita": "Valido (GG/MM/AAAA)",
      "prelievo_data": "Valido (GG/MM/AAAA)",
      "laboratorio_piva": "Valido (numerico)",
      "analisi_cliniche": "Valido (ogni elemento contiene esame, risultato, unita_misura e valori_riferimento)"
    }
  },
  "dato_validato": {
    "laboratorio_nome": "LABORATORIO ANALISI CLINICHE \"VITA NOVA\"",



## 5. Generazione PDF sintetico
Crea un PDF con la stessa struttura dell'originale ma con dati sintetici.

In [15]:
from fpdf import FPDF
import os

record = validated[0]

pdf = FPDF()
pdf.add_page()
pdf.set_auto_page_break(auto=True, margin=15)

# 1. TITOLO DINAMICO: Cerca un campo tipo documento, altrimenti usa "DOCUMENT"
doc_title = str(record.get("document_type", record.get("doc_type", "DOCUMENT"))).upper()

pdf.set_font("Helvetica", "B", 24)
pdf.set_text_color(30, 60, 120)
pdf.cell(0, 12, doc_title, new_x="LMARGIN", new_y="NEXT", align="C")
pdf.ln(4)

pdf.set_draw_color(30, 60, 120)
pdf.set_line_width(0.5)
pdf.line(10, pdf.get_y(), 200, pdf.get_y())
pdf.ln(6)

# SET PER TRACCIARE I CAMPI STAMPATI (La nostra rete di salvataggio)
printed_keys = set()
printed_keys.add("document_type")
printed_keys.add("doc_type")

# Seller / Client
pdf.set_font("Helvetica", "B", 11)
pdf.set_text_color(80, 80, 80)
pdf.cell(95, 7, "SELLER / ISSUER", new_x="RIGHT", new_y="LAST")
pdf.cell(95, 7, "CLIENT / RECIPIENT", new_x="LMARGIN", new_y="NEXT")

pdf.set_draw_color(200, 200, 200)
pdf.line(10, pdf.get_y(), 200, pdf.get_y())
pdf.ln(3)

pdf.set_font("Helvetica", "", 10)
pdf.set_text_color(40, 40, 40)

seller_lines = []
client_lines = []

for fd in extraction.schema_def:
    name = fd.field
    val = str(record.get(name, ""))[:50]
    if not val or val == "None":
        continue
    
    if "seller" in name.lower() or name.lower() in ["company_name", "tax_id", "iban"]:
        seller_lines.append(val)
        printed_keys.add(name)
    elif "client" in name.lower() or "bill_to" in name.lower() or "ship_to" in name.lower():
        client_lines.append(val)
        printed_keys.add(name)

max_lines = max(len(seller_lines), len(client_lines), 1)
for i in range(max_lines):
    s = seller_lines[i] if i < len(seller_lines) else ""
    c = client_lines[i] if i < len(client_lines) else ""
    pdf.cell(95, 6, s, new_x="RIGHT", new_y="LAST")
    pdf.cell(95, 6, c, new_x="LMARGIN", new_y="NEXT")

pdf.ln(4)

# Info documento
pdf.set_font("Helvetica", "B", 10)
pdf.set_text_color(80, 80, 80)
for fd in extraction.schema_def:
    name = fd.field
    if any(kw in name.lower() for kw in ["invoice", "doc_id", "date", "number"]):
        val = str(record.get(name, ""))[:50]
        if val and val != "None":
            pdf.cell(0, 6, f"{name.replace('_', ' ').title()}: {val}", new_x="LMARGIN", new_y="NEXT")
            printed_keys.add(name)

pdf.ln(6)

# Items table
items = record.get("items", record.get("line_items", []))
printed_keys.update(["items", "line_items"]) # Segniamo le tabelle come elaborate
if items:
    col_widths = [12, 62, 18, 18, 28, 26, 26]
    headers = ["#", "Description", "Qty", "UM", "Net Price", "Net Worth", "Gross"]

    pdf.set_fill_color(30, 60, 120)
    pdf.set_text_color(255, 255, 255)
    pdf.set_font("Helvetica", "B", 9)
    for h, w in zip(headers, col_widths):
        pdf.cell(w, 8, h, border=0, fill=True, align="C", new_x="RIGHT", new_y="LAST")
    pdf.ln()

    pdf.set_text_color(40, 40, 40)
    pdf.set_font("Helvetica", "", 9)
    fill = False
    for idx, item in enumerate(items):
        pdf.set_fill_color(245, 245, 250) if fill else pdf.set_fill_color(255, 255, 255)
        row = [
            str(idx + 1),
            str(item.get("description", item.get("name", "")))[:40],
            str(item.get("qty", item.get("quantity", "")))[:8],
            str(item.get("um", item.get("unit", "each")))[:8],
            str(item.get("net_price", item.get("unit_price", "")))[:12],
            str(item.get("net_worth", item.get("net_value", "")))[:12],
            str(item.get("gross_worth", item.get("gross_value", item.get("total", ""))))[:12],
        ]
        for val, w in zip(row, col_widths):
            pdf.cell(w, 7, val, border=0, fill=True, align="C", new_x="RIGHT", new_y="LAST")
        pdf.ln()
        fill = not fill

pdf.ln(4)

# Summary
pdf.set_font("Helvetica", "B", 10)
pdf.set_text_color(80, 80, 80)
pdf.cell(0, 7, "SUMMARY / TOTALS", new_x="LMARGIN", new_y="NEXT")
pdf.set_draw_color(200, 200, 200)
pdf.line(10, pdf.get_y(), 200, pdf.get_y())
pdf.ln(3)

pdf.set_font("Helvetica", "", 10)
pdf.set_text_color(40, 40, 40)
for fd in extraction.schema_def:
    name = fd.field
    if any(kw in name.lower() for kw in ["total", "vat", "tax", "net", "gross", "subtotal", "amount", "summary"]):
        raw_val = record.get(name)
        if not raw_val:
            continue
        
        printed_keys.add(name)
        if isinstance(raw_val, dict):
            for k, v in raw_val.items():
                val_str = str(v)[:40]
                pdf.cell(0, 6, f"{k.replace('_', ' ').title()}: {val_str}", new_x="LMARGIN", new_y="NEXT")
        else:
            val_str = str(raw_val)[:40]
            if val_str and val_str != "None":
                pdf.cell(0, 6, f"{name.replace('_', ' ').title()}: {val_str}", new_x="LMARGIN", new_y="NEXT")

pdf.ln(8)

# ---------------------------------------------------------
# NUOVA SEZIONE: ADDITIONAL DATA (Per documenti non-fattura)
# ---------------------------------------------------------
# Cerchiamo se ci sono campi che NON abbiamo ancora stampato
additional_fields = [fd for fd in extraction.schema_def if fd.field not in printed_keys]

# Filtriamo quelli vuoti
valid_additional_fields = []
for fd in additional_fields:
    val = record.get(fd.field)
    if val and str(val) != "None" and str(val) != "[]" and str(val) != "{}":
        valid_additional_fields.append((fd.field, val))

# Se ci sono campi extra, li stampiamo in una lista generica
if valid_additional_fields:
    pdf.set_font("Helvetica", "B", 10)
    pdf.set_text_color(80, 80, 80)
    pdf.cell(0, 7, "ADDITIONAL INFORMATION", new_x="LMARGIN", new_y="NEXT")
    pdf.set_draw_color(200, 200, 200)
    pdf.line(10, pdf.get_y(), 200, pdf.get_y())
    pdf.ln(3)

    pdf.set_font("Helvetica", "", 10)
    pdf.set_text_color(40, 40, 40)
    
    for name, raw_val in valid_additional_fields:
        if isinstance(raw_val, dict):
            pdf.set_font("Helvetica", "B", 10)
            pdf.cell(0, 6, f"{name.replace('_', ' ').title()}:", new_x="LMARGIN", new_y="NEXT")
            pdf.set_font("Helvetica", "", 10)
            for k, v in raw_val.items():
                val_str = str(v)[:80]
                pdf.cell(0, 6, f"   - {k.replace('_', ' ').title()}: {val_str}", new_x="LMARGIN", new_y="NEXT")
        elif isinstance(raw_val, list):
            pdf.set_font("Helvetica", "B", 10)
            pdf.cell(0, 6, f"{name.replace('_', ' ').title()}:", new_x="LMARGIN", new_y="NEXT")
            pdf.set_font("Helvetica", "", 10)
            for item in raw_val:
                pdf.cell(0, 6, f"   - {str(item)[:80]}", new_x="LMARGIN", new_y="NEXT")
        else:
            val_str = str(raw_val)[:80]
            pdf.cell(0, 6, f"{name.replace('_', ' ').title()}: {val_str}", new_x="LMARGIN", new_y="NEXT")

# Footer
pdf.ln(10)
pdf.set_font("Helvetica", "I", 8)
pdf.set_text_color(150, 150, 150)
pdf.cell(0, 5, "This is a synthetic document generated for testing purposes.", align="C", new_x="LMARGIN", new_y="NEXT")

os.makedirs("output/notebook_v4", exist_ok=True)
pdf_path = "output/notebook_v4/synthetic_DIFFdata.pdf"
pdf.output(pdf_path)
print(f"PDF generato: {pdf_path}")

PDF generato: output/notebook_v4/synthetic_DIFFdata.pdf


## Bonus: workflow completo in un colpo solo

In [ ]:
from inferencev4 import build_graph

app = build_graph()

result = app.invoke({
    "messages": [
        HumanMessage(content="1"),
        HumanMessage(content="run_id:notebook_v4_full"),
    ],
    "raw_text": raw_text,
    "document_schema": "",
    "source_data": "",
    "generated_data": "",
    "validated_data": "",
    "final_data": [],
    "errors": [],
    "pdf_path": "",
})

print(f"Completato: {len(result['final_data'])} record + PDF: {result['pdf_path']}")

2026-05-28 22:23:26.531 | INFO     | inferencev4:analyze_node:92 - [Agente 1] Analisi del documento - deduzione struttura e estrazione dati
2026-05-28 22:28:29.498 | INFO     | inferencev4:analyze_node:116 - Schema dedotto: 25 campi
2026-05-28 22:28:29.506 | INFO     | inferencev4:generate_node:127 - [Agente 2] Generazione di 1 record sintetici
2026-05-28 22:28:46.706 | INFO     | inferencev4:generate_node:156 - Generati 1 record sintetici
2026-05-28 22:28:46.709 | INFO     | inferencev4:validate_node:161 - [Agente 3] Validazione e correzione dei record generati
2026-05-28 22:29:13.504 | INFO     | inferencev4:validate_node:184 - Validazione completata: 1 record
2026-05-28 22:29:13.507 | INFO     | inferencev4:generate_pdf_node:221 - [Agente 5] Generazione PDF sintetico
2026-05-28 22:29:13.622 | INFO     | inferencev4:generate_pdf_node:383 - PDF sintetico salvato: output/notebook_v4_full/synthetic_invoice.pdf
2026-05-28 22:29:13.625 | INFO     | inferencev4:save_node:389 - [Salvataggio

Completato: 1 record + PDF: output/notebook_v4_full/synthetic_invoice.pdf
